# Analyze Hotel Pairs from Delta Lake

This notebook analyzes scored hotel pairs from the Delta Lake table to identify optimal filtering thresholds.

**Data Source**: `s3a://delta-lake/bronze/hotel_pairs/`

## 1. Setup Spark Session with MinIO Configuration

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round, min as spark_min, max as spark_max
import os

# Override any problematic environment variables
os.environ.pop('HADOOP_CONF_DIR', None)
os.environ.pop('YARN_CONF_DIR', None)

# Create Spark session with MinIO/S3A configuration
# Simplified without Delta Lake dependencies - read as parquet directly
spark = SparkSession.builder \
    .appName("Analyze Hotel Pairs") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.send.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3") \
    .config("spark.hadoop.fs.s3a.retry.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.interval", "500") \
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "1000") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "50") \
    .config("spark.hadoop.fs.s3a.threads.max", "10") \
    .config("spark.hadoop.fs.s3a.threads.core", "5") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "5") \
    .config("spark.hadoop.fs.s3a.readahead.range", "65536") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5") \
    .config("spark.hadoop.fs.s3a.list.version", "2") \
    .config("spark.hadoop.fs.s3a.committer.threads", "4") \
    .config("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/staging") \
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.threshold", "2147483647") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "disk") \
    .config("spark.hadoop.fs.s3a.fast.upload.active.blocks", "4") \
    .config("spark.hadoop.fs.s3a.block.size", "33554432") \
    .config("spark.hadoop.fs.s3a.metadatastore.authoritative", "false") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark session created successfully")
print("✓ Spark UI available at: http://localhost:4050")

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/nakul.patil/.ivy2.5.2/cache
The jars for the packages stored in: /Users/nakul.patil/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b20bebd4-158c-42d8-a447-d5d8c6884477;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 109ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final

✓ Spark session created successfully
✓ Spark UI available at: http://localhost:4050


## 2. Load and Explore Hotel Pairs Data

In [8]:
import json
from pyspark.sql.functions import col, lit, when, expr

# Read hotel pairs from Delta Lake (as parquet - Delta is parquet under the hood)
delta_path = "s3a://delta-lake/bronze/hotel_pairs/"
print(f"Reading from: {delta_path}")

# Read as parquet directly to avoid Delta Lake dependency issues
df = spark.read.format("parquet").load(delta_path)

print(f"\n✓ Data loaded successfully")
print(f"Total pairs: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

# 1. Print Schema
print("\n" + "="*80)
print("SCHEMA:")
print("="*80)
df.printSchema()

# 2. Print Sample Records
print("\n" + "="*80)
print("SAMPLE RECORDS (showing 10):")
print("="*80)
df.show(10, truncate=True)

# Get score columns - ALL score columns from the processor
score_columns = [
    'geo_distance_km',
    'name_score_jaccard', 'normalized_name_score_jaccard',
    'name_score_lcs', 'normalized_name_score_lcs',
    'name_score_levenshtein', 'normalized_name_score_levenshtein',
    'name_score_sbert', 'normalized_name_score_sbert',
    'name_score_containment', 'normalized_name_score_containment',
    'average_name_score', 'average_normalized_name_score',
    'address_line1_score', 'address_sbert_score',
    'postal_code_match', 'country_match',
    'phone_match_score', 'email_match_score', 'fax_match_score',
    'star_ratings_score',
    'property_type_score', 'name_unit_score', 'address_unit_score', 'supplier_score', 'overall_pair_score'
]

# Filter to only columns that exist
existing_score_cols = [c for c in score_columns if c in df.columns]

print("\n" + "="*80)
print(f"SCORE COLUMNS AVAILABLE ({len(existing_score_cols)}):")
print("="*80)
for col_name in existing_score_cols:
    print(f"  - {col_name}")

# 3. Analyze Score Distributions for Different Match Types
print("\n" + "="*80)
print("ANALYZING SCORE DISTRIBUTIONS:")
print("="*80)

# Check if we have id columns to identify same entities (old cluster IDs)
has_id = 'id_i' in df.columns and 'id_j' in df.columns

if has_id:
    # Same entity pairs (id_i == id_j) - these should be perfect matches
    # Note: id_i and id_j are old cluster IDs, not hotel IDs
    same_entity_df = df.filter(col("id_i") == col("id_j"))
    different_entity_df = df.filter(col("id_i") != col("id_j"))
    
    same_count = same_entity_df.count()
    diff_count = different_entity_df.count()
    
    if same_count > 0:
        print(f"\n📊 Same Entity Pairs (id_i == id_j): {same_count:,}")
        print(f"📊 Different Entity Pairs (id_i != id_j): {diff_count:,}")
        print("Same-entity pairs represent identical hotels (same old cluster ID) and should have high scores.")
        
        # Calculate statistics for BOTH groups to find separation
        print("\n" + "="*80)
        print("FINDING SEPARABLE SCORE DIMENSIONS:")
        print("="*80)
        print("Goal: Find scores where same-entity MIN > different-entity MAX (perfect separation)")
        
        separable_scores = []
        stats_combined = []
        
        for score_col in existing_score_cols:
            # Same-entity statistics - KEY PERCENTILES: MIN, MAX, P50, P90, P95, P99
            same_stats = same_entity_df.agg(
                spark_round(spark_min(col(score_col)), 4).alias('same_min'),
                spark_round(spark_max(col(score_col)), 4).alias('same_max'),
                expr(f"percentile_approx({score_col}, 0.50)").alias('same_p50'),
                expr(f"percentile_approx({score_col}, 0.90)").alias('same_p90'),
                expr(f"percentile_approx({score_col}, 0.95)").alias('same_p95'),
                expr(f"percentile_approx({score_col}, 0.99)").alias('same_p99')
            ).collect()[0]
            
            # Different-entity statistics - KEY PERCENTILES: MIN, MAX, P50, P90, P95, P99
            diff_stats = different_entity_df.agg(
                spark_round(spark_min(col(score_col)), 4).alias('diff_min'),
                spark_round(spark_max(col(score_col)), 4).alias('diff_max'),
                expr(f"percentile_approx({score_col}, 0.50)").alias('diff_p50'),
                expr(f"percentile_approx({score_col}, 0.90)").alias('diff_p90'),
                expr(f"percentile_approx({score_col}, 0.95)").alias('diff_p95'),
                expr(f"percentile_approx({score_col}, 0.99)").alias('diff_p99')
            ).collect()[0]
            
            same_min = float(same_stats['same_min']) if same_stats['same_min'] is not None else 0.0
            same_max = float(same_stats['same_max']) if same_stats['same_max'] is not None else 0.0
            same_p50 = float(same_stats['same_p50']) if same_stats['same_p50'] is not None else 0.0
            same_p90 = float(same_stats['same_p90']) if same_stats['same_p90'] is not None else 0.0
            same_p95 = float(same_stats['same_p95']) if same_stats['same_p95'] is not None else 0.0
            same_p99 = float(same_stats['same_p99']) if same_stats['same_p99'] is not None else 0.0
            diff_min = float(diff_stats['diff_min']) if diff_stats['diff_min'] is not None else 0.0
            diff_max = float(diff_stats['diff_max']) if diff_stats['diff_max'] is not None else 0.0
            diff_p50 = float(diff_stats['diff_p50']) if diff_stats['diff_p50'] is not None else 0.0
            diff_p90 = float(diff_stats['diff_p90']) if diff_stats['diff_p90'] is not None else 0.0
            diff_p95 = float(diff_stats['diff_p95']) if diff_stats['diff_p95'] is not None else 0.0
            diff_p99 = float(diff_stats['diff_p99']) if diff_stats['diff_p99'] is not None else 0.0
            
            # Calculate separation gap
            # For distance metrics (lower is better), flip the logic
            is_distance_metric = score_col == 'geo_distance_km'
            
            if is_distance_metric:
                # For distance: same MAX should be < different MIN
                gap = diff_min - same_max
                separation_quality = "PERFECT" if gap > 0 else f"OVERLAP (gap: {gap:.4f})"
            else:
                # For similarity: same MIN should be > different MAX
                gap = same_min - diff_max
                separation_quality = "PERFECT" if gap > 0 else f"OVERLAP (gap: {gap:.4f})"
            
            stats_combined.append({
                'column': score_col,
                'is_distance_metric': is_distance_metric,
                'same_min': same_min,
                'same_max': same_max,
                'same_p50': same_p50,
                'same_p90': same_p90,
                'same_p95': same_p95,
                'same_p99': same_p99,
                'diff_min': diff_min,
                'diff_max': diff_max,
                'diff_p50': diff_p50,
                'diff_p90': diff_p90,
                'diff_p95': diff_p95,
                'diff_p99': diff_p99,
                'gap': gap,
                'separable': gap > 0
            })
            
            print(f"\n{score_col}:")
            if is_distance_metric:
                print(f"  Same-entity:  MIN={same_min:.4f}  MAX={same_max:.4f}  P50={same_p50:.4f}  P90={same_p90:.4f}  P95={same_p95:.4f}  P99={same_p99:.4f}")
                print(f"  Diff-entity:  MIN={diff_min:.4f}  MAX={diff_max:.4f}  P50={diff_p50:.4f}  P90={diff_p90:.4f}  P95={diff_p95:.4f}  P99={diff_p99:.4f}")
                print(f"  Separable: {separation_quality}")
                if gap > 0:
                    threshold = (same_max + diff_min) / 2
                    print(f"  ✓ OPTIMAL THRESHOLD: {score_col} <= {threshold:.4f}")
                    separable_scores.append((score_col, '<=', round(threshold, 4)))
            else:
                print(f"  Same-entity:  MIN={same_min:.4f}  MAX={same_max:.4f}  P50={same_p50:.4f}  P90={same_p90:.4f}  P95={same_p95:.4f}  P99={same_p99:.4f}")
                print(f"  Diff-entity:  MIN={diff_min:.4f}  MAX={diff_max:.4f}  P50={diff_p50:.4f}  P90={diff_p90:.4f}  P95={diff_p95:.4f}  P99={diff_p99:.4f}")
                print(f"  Separable: {separation_quality}")
                if gap > 0:
                    threshold = (same_min + diff_max) / 2
                    print(f"  ✓ OPTIMAL THRESHOLD: {score_col} >= {threshold:.4f}")
                    separable_scores.append((score_col, '>=', round(threshold, 4)))
        
        # 4. Create WHERE clauses based on separability analysis
        print("\n" + "="*80)
        print("BUILDING OPTIMAL WHERE CLAUSES:")
        print("="*80)
        
        if len(separable_scores) > 0:
            print(f"\n✅ Found {len(separable_scores)} perfectly separable score dimensions!")
            print("These can achieve 100% precision and 100% recall individually.\n")
            
            for score_col, op, threshold in separable_scores:
                print(f"  • {score_col} {op} {threshold}")
            
            # Create WHERE clauses
            # Option 1: Use ANY separable dimension (OR logic) - most permissive
            or_conditions = [f"{col} {op} {thresh}" for col, op, thresh in separable_scores]
            sql_where_any = " OR ".join(or_conditions)
            
            # Option 2: Use ALL separable dimensions (AND logic) - most strict
            sql_where_all = " AND ".join(or_conditions)
            
            # Option 3: Use best single dimension (highest gap)
            best_score = sorted([(abs(s['gap']), s) for s in stats_combined if s['separable']], reverse=True)[0][1]
            if best_score['is_distance_metric']:
                # For distance, use a threshold less than same_max
                threshold_best = round(best_score['same_max'] * 1.01, 4)  # Slight buffer
                sql_where_best = f"{best_score['column']} <= {threshold_best}"
            else:
                # For similarity, use a threshold greater than same_min
                threshold_best = round(best_score['same_min'] * 0.99, 4)  # Slight buffer  
                sql_where_best = f"{best_score['column']} >= {threshold_best}"
            
            print(f"\n🎯 BEST SINGLE DIMENSION (largest gap={best_score['gap']:.4f}):")
            print(f"   {best_score['column']}")
            
        else:
            # No perfect separation - use percentile-based approach
            print("\n⚠️  No perfectly separable dimensions found!")
            print("Using percentile-based approach to maximize recall while minimizing false positives.\n")
            
            # Strategy: Use MIN-based thresholds for maximum recall
            threshold_conditions = []
            
            for stat in stats_combined:
                if stat['is_distance_metric']:
                    # Use max of same-entity as threshold
                    threshold = round(stat['same_max'] * 1.1, 4)
                    threshold_conditions.append(f"{stat['column']} <= {threshold}")
                else:
                    # Use MIN of same-entity for 100% recall
                    threshold = round(stat['same_min'] * 0.98, 4) if stat['same_min'] > 0 else 0.0
                    if threshold > 0:
                        threshold_conditions.append(f"{stat['column']} >= {threshold}")
            
            # Create multi-dimensional WHERE clauses
            # Combine name + geo + address for best discrimination
            name_conditions = [c for c in threshold_conditions if 'name_score' in c or 'average_name' in c]
            geo_conditions = [c for c in threshold_conditions if 'geo_distance' in c]
            addr_conditions = [c for c in threshold_conditions if 'address' in c]
            
            sql_where_best = " AND ".join(name_conditions[:2] + geo_conditions + addr_conditions[:1]) if name_conditions else "1=1"
            sql_where_any = " OR ".join(name_conditions[:1] + geo_conditions) if name_conditions else "1=1"
            sql_where_all = " AND ".join(threshold_conditions[:5]) if threshold_conditions else "1=1"
            
            separable_scores = []  # Empty since no perfect separation
        
        # 5. Save configurations to JSON file
        output_config = {
            'analysis_date': '2026-03-03',
            'total_pairs': df.count(),
            'same_entity_pairs': same_count,
            'different_entity_pairs': diff_count,
            'note': 'id_i and id_j are old cluster IDs used to identify same entities',
            'score_columns': existing_score_cols,
            'separable_scores': [(col, op, thresh) for col, op, thresh in separable_scores],
            'stats_by_score': stats_combined,
            'where_clauses': {
                'best_single': sql_where_best,
                'all_conditions': sql_where_all,
                'any_condition': sql_where_any
            },
            'recommended': 'best_single' if len(separable_scores) > 0 else 'all_conditions'
        }
        
        output_path = '/Users/nakul.patil/Documents/hotel-mapping/output/pair_score_thresholds.json'
        with open(output_path, 'w') as f:
            json.dump(output_config, f, indent=2)
        
        print(f"\n✓ Configuration saved to: {output_path}")
        
        print("\n" + "="*80)
        print("WHERE CLAUSE OPTIONS:")
        print("="*80)
        print(f"\n1️⃣  BEST SINGLE (recommended if separable):")
        print(f"    {sql_where_best}")
        print(f"\n2️⃣  ALL CONDITIONS (strictest - AND logic):")
        print(f"    {sql_where_all}")
        print(f"\n3️⃣  ANY CONDITION (most permissive - OR logic):")
        print(f"    {sql_where_any}")
        
    else:
        print("⚠️ No same-entity pairs found (id_i == id_j)")
        output_config = {
            'error': 'No same-entity pairs available for analysis',
            'total_pairs': df.count()
        }
        output_path = '/Users/nakul.patil/Documents/hotel-mapping/output/pair_score_thresholds.json'
        with open(output_path, 'w') as f:
            json.dump(output_config, f, indent=2)
else:
    print("⚠️ id_i and id_j columns not found - cannot identify same entities")
    output_config = {
        'error': 'id columns not available',
        'total_pairs': df.count()
    }
    output_path = '/Users/nakul.patil/Documents/hotel-mapping/output/pair_score_thresholds.json'
    with open(output_path, 'w') as f:
        json.dump(output_config, f, indent=2)

print("\n✓ Analysis complete!")


Reading from: s3a://delta-lake/bronze/hotel_pairs/

✓ Data loaded successfully
Total pairs: 27,266
Total columns: 34

SCHEMA:
root
 |-- id_i: string (nullable = true)
 |-- id_j: string (nullable = true)
 |-- providerHotelId_i: string (nullable = true)
 |-- providerHotelId_j: string (nullable = true)
 |-- name_i: string (nullable = true)
 |-- name_j: string (nullable = true)
 |-- uid_i: string (nullable = true)
 |-- uid_j: string (nullable = true)
 |-- normalized_name_i: string (nullable = true)
 |-- normalized_name_j: string (nullable = true)
 |-- combined_address_i: string (nullable = true)
 |-- combined_address_j: string (nullable = true)
 |-- geo_distance_km: double (nullable = true)
 |-- name_score_jaccard: float (nullable = true)
 |-- normalized_name_score_jaccard: float (nullable = true)
 |-- name_score_lcs: float (nullable = true)
 |-- normalized_name_score_lcs: float (nullable = true)
 |-- name_score_levenshtein: float (nullable = true)
 |-- normalized_name_score_levenshtein: f

## 3. Knowledgebase: Achieving 100% Precision + 100% Recall

### Problem Statement
We have 2,194 same-entity pairs (id_i == id_j) out of 6,904 total pairs. Goal: Find a WHERE clause based on score columns that captures **exactly** these 2,194 pairs and **none** of the 4,710 different-entity pairs.

### Options to Achieve 100% P + 100% R:

#### Option 1: Perfect Score Separation (Ideal)
**Condition**: Find a score dimension where `min(same-entity) > max(different-entity)`
- If such a score exists, set threshold = `(min_same + max_diff) / 2`
- This guarantees 100% P + 100% R mathematically
- **Status**: Check Cell 4 output to see if any score has "PERFECT" separation

#### Option 2: Composite Score with Linear Combination
**Approach**: Combine multiple scores with weights to maximize separation
```python
composite_score = w1*name_score + w2*geo_score + w3*address_score + ...
```
- Use statistical methods to find optimal weights
- May achieve separation even if individual scores don't
- **Implementation**: Train a simple linear model or use ROC curve analysis

#### Option 3: Use id_i == id_j Directly (Cheating)
**Reality Check**: Filtering by `id_i == id_j` gives perfect P+R but defeats the purpose
- This is what we're trying to replicate with scores
- Useful only to verify data integrity

#### Option 4: Machine Learning Classifier
**Approach**: Train a model on labeled data
- Features: All score columns
- Label: 1 if id_i == id_j, 0 otherwise
- Models: Logistic Regression, Random Forest, XGBoost
- Can learn complex decision boundaries that simple thresholds can't capture

#### Option 5: Iterative Threshold Refinement
**Approach**: Build a complex rule with AND/OR logic
```sql
(score1 >= threshold1 AND score2 >= threshold2) OR 
(score1 >= threshold3 AND score3 <= threshold4)
```
- Test all combinations of score thresholds
- Find the rule that exactly captures 2,194 pairs
- Computationally expensive but guaranteed to find solution if it exists

#### Option 6: Accept Trade-off (Pragmatic)
**Reality**: If score distributions overlap significantly:
- 100% Recall + High Precision (few false positives) → More useful in practice
- Perfect separation may not exist with current score dimensions
- Consider adding more features (e.g., contact info similarity, business type)

### Recommended Approach:
1. **First**: Check if Option 1 works (Cell 4 analysis)
2. **If not**: Try Option 2 (composite score) → Create in next cell
3. **If still not**: Try Option 4 (ML classifier) → More complex but powerful
4. **Fallback**: Accept 100% Recall + 95-99% Precision as success

### Key Insight:
The existence of 4,710 false positives with current WHERE clauses suggests **significant score overlap** between same-entity and different-entity pairs. This means:
- Some same-entity pairs have LOW scores (perhaps due to data quality issues)
- Some different-entity pairs have HIGH scores (perhaps legitimate duplicates or similar hotels)
- Simple threshold-based filtering may be fundamentally limited


In [ ]:
import json

# Read saved configuration
config_path = '/Users/nakul.patil/Documents/hotel-mapping/output/pair_score_thresholds.json'
with open(config_path, 'r') as f:
    config = json.load(f)

print("Loaded configuration:")
print(json.dumps(config, indent=2))

# Test all WHERE clause options
df.createOrReplaceTempView("hotel_pairs")

if 'where_clauses' in config:
    same_entity_total = df.filter(col("id_i") == col("id_j")).count()
    diff_entity_total = df.filter(col("id_i") != col("id_j")).count()
    
    print("\n" + "="*80)
    print("TESTING ALL WHERE CLAUSE OPTIONS:")
    print("="*80)
    print(f"Ground Truth: {same_entity_total:,} same-entity pairs, {diff_entity_total:,} different-entity pairs")
    print(f"Target: 100% Precision (0 false positives) + 100% Recall (all {same_entity_total:,} captured)")
    
    results = {}
    
    # Test each WHERE clause option
    for option_name in ['best_single', 'all_conditions', 'any_condition']:
        if option_name not in config['where_clauses']:
            continue
            
        where_clause = config['where_clauses'][option_name]
        print(f"\n{'='*80}")
        print(f"Testing: {option_name.replace('_', ' ').upper()}")
        print(f"{'='*80}")
        print(f"WHERE: {where_clause}")
        
        # Apply filter
        try:
            filtered_df = spark.sql(f"SELECT * FROM hotel_pairs WHERE {where_clause}")
            
            # Calculate metrics
            total_captured = filtered_df.count()
            same_captured = filtered_df.filter(col("id_i") == col("id_j")).count()
            diff_captured = filtered_df.filter(col("id_i") != col("id_j")).count()
            
            same_missed = same_entity_total - same_captured
            
            # Recall = how many same-entity pairs we captured
            recall = 100 * same_captured / same_entity_total if same_entity_total > 0 else 0
            
            # Precision = what % of captured pairs are same-entity
            precision = 100 * same_captured / total_captured if total_captured > 0 else 0
            
            # False positive rate = how many different-entity pairs leaked through
            fpr = 100 * diff_captured / diff_entity_total if diff_entity_total > 0 else 0
            
            results[option_name] = {
                'total_captured': total_captured,
                'same_captured': same_captured,
                'diff_captured': diff_captured,
                'same_missed': same_missed,
                'recall': recall,
                'precision': precision,
                'fpr': fpr
            }
            
            print(f"\n📊 Results:")
            print(f"   Total captured: {total_captured:,}")
            print(f"   ├─ Same-entity (id_i == id_j): {same_captured:,} ✓")
            print(f"   └─ Different-entity (id_i != id_j): {diff_captured:,} {'✅' if diff_captured == 0 else '⚠️'}")
            print(f"\n📈 Metrics:")
            print(f"   Recall: {recall:.2f}% ({same_captured}/{same_entity_total}) {'✅' if recall == 100 else '❌'}")
            print(f"   Precision: {precision:.2f}% ({same_captured}/{total_captured}) {'✅' if precision == 100 else '❌'}")
            print(f"   False Positive Rate: {fpr:.2f}% ({diff_captured}/{diff_entity_total}) {'✅' if fpr == 0 else '⚠️'}")
            print(f"   Same-entity MISSED: {same_missed:,} {'✅' if same_missed == 0 else '❌'}")
            
            # Show quality assessment
            if same_missed == 0 and diff_captured == 0:
                print(f"\n🏆 PERFECT: 100% Precision + 100% Recall!")
            elif same_missed == 0 and fpr < 1:
                print(f"\n✅ EXCELLENT: 100% Recall, <1% false positives")
            elif same_missed == 0 and fpr < 5:
                print(f"\n✅ VERY GOOD: 100% Recall, <5% false positives")
            elif same_missed == 0:
                print(f"\n✓ GOOD: 100% Recall achieved")
            elif recall >= 99:
                print(f"\n⚠️  HIGH RECALL: {recall:.1f}%, missing only {same_missed}")
            else:
                print(f"\n❌ NEEDS IMPROVEMENT: Missing {same_missed} same-entity pairs")
        
        except Exception as e:
            print(f"\n❌ ERROR: {str(e)}")
            results[option_name] = None
    
    # Find best option
    print("\n" + "="*80)
    print("RECOMMENDATION:")
    print("="*80)
    
    # Best option: 100% recall and 100% precision (if possible), else highest recall with best precision
    best_option = None
    best_score = -1
    
    for opt in results.keys():
        if results[opt] is None:
            continue
        r = results[opt]
        
        # Perfect score: recall=100 and precision=100
        if r['recall'] == 100 and r['precision'] == 100:
            best_option = opt
            break
        
        # Score: prioritize recall, then precision
        score = r['recall'] + (r['precision'] * 0.5)
        if score > best_score:
            best_score = score
            best_option = opt
    
    if best_option and results[best_option]:
        best = results[best_option]
        print(f"\n🎯 BEST OPTION: {best_option.replace('_', ' ').upper()}")
        print(f"   WHERE: {config['where_clauses'][best_option]}")
        print(f"   Recall: {best['recall']:.2f}%, Precision: {best['precision']:.2f}%, FPR: {best['fpr']:.2f}%")
        print(f"   Captures: {best['same_captured']:,} same-entity + {best['diff_captured']:,} different-entity = {best['total_captured']:,} total")
        
        if best['recall'] == 100 and best['precision'] == 100:
            print(f"\n🏆 PERFECT SEPARATION ACHIEVED!")
        elif best['recall'] == 100:
            print(f"\n✅ All same-entity pairs captured! ({best['diff_captured']} false positives to investigate)")
        
        # Show samples
        best_filtered_df = spark.sql(f"SELECT * FROM hotel_pairs WHERE {config['where_clauses'][best_option]}")
        
        print(f"\n" + "="*80)
        print(f"SAMPLE ANALYSIS FOR {best_option.replace('_', ' ').upper()} OPTION:")
        print("="*80)
        
        # Sample 1: Same-entity pairs captured (GOOD)
        same_captured_sample = best_filtered_df.filter(col("id_i") == col("id_j")).count()
        if same_captured_sample > 0:
            print(f"\n1️⃣  Same-entity pairs CAPTURED (id_i == id_j) - TRUE POSITIVES: {same_captured_sample:,}")
            best_filtered_df.filter(col("id_i") == col("id_j")).select(
                'id_i', 'id_j', 'name_i', 'name_j',
                'average_name_score', 'geo_distance_km', 'address_line1_score'
            ).show(5, truncate=True)
        
        # Sample 2: Same-entity pairs MISSED (BAD)
        same_missed_count = best['same_missed']
        if same_missed_count > 0:
            print(f"\n2️⃣  Same-entity pairs MISSED (id_i == id_j) - FALSE NEGATIVES: {same_missed_count:,} ❌")
            print("    These are critical failures - perfect matches that our thresholds rejected!")
            missed_df = df.filter(col("id_i") == col("id_j")).subtract(
                best_filtered_df.filter(col("id_i") == col("id_j"))
            )
            missed_df.select(
                'id_i', 'id_j', 'name_i', 'name_j',
                'average_name_score', 'geo_distance_km',
                'address_line1_score', 'country_match'
            ).show(10, truncate=True)
        else:
            print(f"\n2️⃣  Same-entity pairs MISSED: 0 ✅ Perfect recall!")
        
        # Sample 3: Different-entity pairs captured (INVESTIGATE)
        diff_captured_count = best['diff_captured']
        if diff_captured_count > 0:
            print(f"\n3️⃣  Different-entity pairs CAPTURED (id_i != id_j) - FALSE POSITIVES: {diff_captured_count:,}")
            print("    These might be legitimate duplicates OR they're errors that reduce precision")
            best_filtered_df.filter(col("id_i") != col("id_j")).select(
                'id_i', 'id_j', 'name_i', 'name_j',
                'average_name_score', 'geo_distance_km',
                'address_line1_score', 'country_match'
            ).orderBy(col("average_name_score").desc()).show(15, truncate=True)
        else:
            print(f"\n3️⃣  Different-entity pairs CAPTURED: 0 ✅ Perfect precision!")
    else:
        print("\n⚠️  No valid WHERE clause produced results")
    
else:
    print("⚠️ No filter configuration available")


Loaded configuration:
{
  "analysis_date": "2026-03-03",
  "total_pairs": 27266,
  "same_entity_pairs": 595,
  "different_entity_pairs": 26671,
  "note": "id_i and id_j are old cluster IDs used to identify same entities",
  "score_columns": [
    "geo_distance_km",
    "name_score_jaccard",
    "normalized_name_score_jaccard",
    "name_score_lcs",
    "normalized_name_score_lcs",
    "name_score_levenshtein",
    "normalized_name_score_levenshtein",
    "name_score_sbert",
    "normalized_name_score_sbert",
    "address_line1_score",
    "address_sbert_score",
    "postal_code_match",
    "country_match",
    "phone_match_score",
    "email_match_score",
    "fax_match_score",
    "star_ratings_score",
    "property_type_score",
    "name_unit_score",
    "address_unit_score",
    "supplier_score",
    "overall_pair_score"
  ],
  "separable_scores": [],
  "stats_by_score": [
    {
      "column": "geo_distance_km",
      "is_distance_metric": true,
      "same_min": 0.0,
      "same_m

{"ts": "2026-03-08 18:52:12.800", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `average_name_score` cannot be resolved. Did you mean one of the following? [`overall_pair_score`, `email_match_score`, `name_unit_score`, `address_line1_score`, `address_unit_score`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor84.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o6247.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `average_name_score` cannot be resolved. Did you mean one of the following? [`overall_pair_score`, `email_match_score`, `name_unit_score`, `address_line1_score`, `address_unit_score`]. SQLSTATE: 42703;\n'Pro


📊 Results:
   Total captured: 27,266
   ├─ Same-entity (id_i == id_j): 595 ✓
   └─ Different-entity (id_i != id_j): 26,671 ⚠️

📈 Metrics:
   Recall: 100.00% (595/595) ✅
   Precision: 2.18% (595/27266) ❌
   False Positive Rate: 100.00% (26671/26671) ⚠️
   Same-entity MISSED: 0 ✅

✓ GOOD: 100% Recall achieved

RECOMMENDATION:

🎯 BEST OPTION: ALL CONDITIONS
   WHERE: geo_distance_km <= 0.5293 AND name_score_lcs >= 0.0555 AND name_score_levenshtein >= 0.245 AND name_score_sbert >= 0.3799 AND address_line1_score >= 0.0686
   Recall: 100.00%, Precision: 5.15%, FPR: 41.12%
   Captures: 595 same-entity + 10,966 different-entity = 11,561 total

✅ All same-entity pairs captured! (10966 false positives to investigate)

SAMPLE ANALYSIS FOR ALL CONDITIONS OPTION:

1️⃣  Same-entity pairs CAPTURED (id_i == id_j) - TRUE POSITIVES: 595


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `average_name_score` cannot be resolved. Did you mean one of the following? [`overall_pair_score`, `email_match_score`, `name_unit_score`, `address_line1_score`, `address_unit_score`]. SQLSTATE: 42703;
'Project [id_i#21376, id_j#21377, name_i#21380, name_j#21381, 'average_name_score, geo_distance_km#21388, address_line1_score#21398]
+- Filter (id_i#21376 = id_j#21377)
   +- Project [id_i#21376, id_j#21377, providerHotelId_i#21378, providerHotelId_j#21379, name_i#21380, name_j#21381, uid_i#21382, uid_j#21383, normalized_name_i#21384, normalized_name_j#21385, combined_address_i#21386, combined_address_j#21387, geo_distance_km#21388, name_score_jaccard#21389, normalized_name_score_jaccard#21390, name_score_lcs#21391, normalized_name_score_lcs#21392, name_score_levenshtein#21393, normalized_name_score_levenshtein#21394, name_score_sbert#21395, normalized_name_score_sbert#21396, star_r

26/03/08 20:30:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 908744 ms exceeds timeout 120000 ms
26/03/08 20:30:00 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/08 20:30:01 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

## 4. Comparative Distribution Analysis: Matching vs Non-Matching Entities

This cell provides detailed statistical comparison of score distributions to identify overlaps and refine thresholds.

In [ ]:
from pyspark.sql.functions import col, expr
import pandas as pd
import builtins  # Ensure we can use Python's built-in min/max

print("="*100)
print("COMPARATIVE DISTRIBUTION ANALYSIS: MATCHING vs NON-MATCHING ENTITIES")
print("="*100)
print("\nObjective: Identify score ranges where distributions are:")
print("  • MUTUALLY EXCLUSIVE → Clear threshold possible (100% P+R)")
print("  • OVERLAPPING → Need refined thresholds or composite scores")
print("\n" + "="*100)

# Split data into two groups
same_entity_df = df.filter(col("id_i") == col("id_j"))
diff_entity_df = df.filter(col("id_i") != col("id_j"))

same_count = same_entity_df.count()
diff_count = diff_entity_df.count()

print(f"\n📊 Dataset Split:")
print(f"   Matching Entities (id_i == id_j): {same_count:,} pairs")
print(f"   Non-Matching Entities (id_i != id_j): {diff_count:,} pairs")
print(f"   Total: {same_count + diff_count:,} pairs")

# Get score columns - ALL score columns from the processor
score_columns = [
    'geo_distance_km',
    'name_score_jaccard', 'normalized_name_score_jaccard',
    'name_score_lcs', 'normalized_name_score_lcs',
    'name_score_levenshtein', 'normalized_name_score_levenshtein',
    'name_score_sbert', 'normalized_name_score_sbert',
    'name_score_containment', 'normalized_name_score_containment',
    'average_name_score', 'average_normalized_name_score',
    'address_line1_score', 'address_sbert_score',
    'postal_code_match', 'country_match',
    'phone_match_score', 'email_match_score', 'fax_match_score',
    'star_ratings_score',
    'property_type_score', 'name_unit_score', 'address_unit_score', 'supplier_score'
]

# Filter to existing columns
existing_score_cols = [c for c in score_columns if c in df.columns]

# Filter out binary match scores that have high gaps but aren't useful for thresholding
binary_match_scores = ['fax_match_score', 'email_match_score', 'phone_match_score', 
                        'country_match', 'postal_code_match']
useful_score_cols = [c for c in existing_score_cols if c not in binary_match_scores]

print(f"\n📈 Analyzing {len(useful_score_cols)} score dimensions (excluding {len(binary_match_scores)} binary match scores)...")
print(f"   Excluded: {', '.join([s for s in binary_match_scores if s in existing_score_cols])}")

# Collect detailed statistics for each score
comparison_results = []

for score_col in useful_score_cols:
    # Calculate percentiles and stats for MATCHING entities - KEY PERCENTILES
    same_stats = same_entity_df.agg(
        spark_min(col(score_col)).alias('min'),
        spark_max(col(score_col)).alias('max'),
        expr(f"percentile_approx({score_col}, 0.50)").alias('p50'),
        expr(f"percentile_approx({score_col}, 0.90)").alias('p90'),
        expr(f"percentile_approx({score_col}, 0.95)").alias('p95'),
        expr(f"percentile_approx({score_col}, 0.99)").alias('p99')
    ).collect()[0]
    
    # Calculate percentiles and stats for NON-MATCHING entities - KEY PERCENTILES
    diff_stats = diff_entity_df.agg(
        spark_min(col(score_col)).alias('min'),
        spark_max(col(score_col)).alias('max'),
        expr(f"percentile_approx({score_col}, 0.50)").alias('p50'),
        expr(f"percentile_approx({score_col}, 0.90)").alias('p90'),
        expr(f"percentile_approx({score_col}, 0.95)").alias('p95'),
        expr(f"percentile_approx({score_col}, 0.99)").alias('p99')
    ).collect()[0]
    
    # Determine if distance metric (lower is better) or similarity (higher is better)
    is_distance = score_col == 'geo_distance_km'
    
    # Calculate overlap metrics
    if is_distance:
        # For distance: check if same_max < diff_min (mutually exclusive)
        gap = float(diff_stats['min']) - float(same_stats['max'])
        overlap_type = "MUTUALLY EXCLUSIVE" if gap > 0 else "OVERLAPPING"
        optimal_threshold = round((float(same_stats['max']) + float(diff_stats['min'])) / 2, 4) if gap > 0 else None
        threshold_operator = "<="
        
        # Calculate overlap range
        if gap <= 0:
            overlap_start = builtins.min(float(same_stats['max']), float(diff_stats['max']))
            overlap_end = builtins.max(float(same_stats['min']), float(diff_stats['min']))
            overlap_pct_same = 100.0  # Need more detailed calculation
            overlap_pct_diff = 100.0
        else:
            overlap_start = None
            overlap_end = None
            overlap_pct_same = 0.0
            overlap_pct_diff = 0.0
    else:
        # For similarity: check if same_min > diff_max (mutually exclusive)
        gap = float(same_stats['min']) - float(diff_stats['max'])
        overlap_type = "MUTUALLY EXCLUSIVE" if gap > 0 else "OVERLAPPING"
        optimal_threshold = round((float(same_stats['min']) + float(diff_stats['max'])) / 2, 4) if gap > 0 else None
        threshold_operator = ">="
        
        # Calculate overlap range
        if gap <= 0:
            overlap_start = builtins.max(float(same_stats['min']), float(diff_stats['min']))
            overlap_end = builtins.min(float(same_stats['max']), float(diff_stats['max']))
            overlap_pct_same = 100.0
            overlap_pct_diff = 100.0
        else:
            overlap_start = None
            overlap_end = None
            overlap_pct_same = 0.0
            overlap_pct_diff = 0.0
    
    comparison_results.append({
        'score': score_col,
        'is_distance': is_distance,
        'same_min': float(same_stats['min']),
        'same_max': float(same_stats['max']),
        'same_p50': float(same_stats['p50']),
        'same_p90': float(same_stats['p90']),
        'same_p95': float(same_stats['p95']),
        'same_p99': float(same_stats['p99']),
        'diff_min': float(diff_stats['min']),
        'diff_max': float(diff_stats['max']),
        'diff_p50': float(diff_stats['p50']),
        'diff_p90': float(diff_stats['p90']),
        'diff_p95': float(diff_stats['p95']),
        'diff_p99': float(diff_stats['p99']),
        'gap': gap,
        'overlap_type': overlap_type,
        'optimal_threshold': optimal_threshold,
        'threshold_operator': threshold_operator,
        'overlap_start': overlap_start,
        'overlap_end': overlap_end
    })

# Print detailed comparison for each score
print("\n" + "="*100)
print("DETAILED SCORE COMPARISONS:")
print("="*100)

mutually_exclusive_scores = []
overlapping_scores = []

for result in comparison_results:
    score = result['score']
    print(f"\n{'─'*100}")
    print(f"📊 {score}")
    print(f"{'─'*100}")
    
    if result['is_distance']:
        print(f"Type: DISTANCE METRIC (lower is better) - threshold should be <=")
    else:
        print(f"Type: SIMILARITY METRIC (higher is better) - threshold should be >=")
    
    print(f"\n  MATCHING Entities (id_i == id_j):")
    print(f"    Range: MIN={result['same_min']:.4f} → MAX={result['same_max']:.4f}")
    print(f"    Percentiles: P50={result['same_p50']:.4f} | P90={result['same_p90']:.4f} | P95={result['same_p95']:.4f} | P99={result['same_p99']:.4f}")
    
    print(f"\n  NON-MATCHING Entities (id_i != id_j):")
    print(f"    Range: MIN={result['diff_min']:.4f} → MAX={result['diff_max']:.4f}")
    print(f"    Percentiles: P50={result['diff_p50']:.4f} | P90={result['diff_p90']:.4f} | P95={result['diff_p95']:.4f} | P99={result['diff_p99']:.4f}")
    
    if result['overlap_type'] == "MUTUALLY EXCLUSIVE":
        print(f"\n  ✅ STATUS: {result['overlap_type']}")
        print(f"     Gap: {result['gap']:.4f} (distributions don't overlap)")
        print(f"     🎯 OPTIMAL THRESHOLD: {score} {result['threshold_operator']} {result['optimal_threshold']}")
        print(f"     This threshold guarantees 100% Precision + 100% Recall!")
        mutually_exclusive_scores.append((score, result['threshold_operator'], result['optimal_threshold']))
    else:
        print(f"\n  ⚠️  STATUS: {result['overlap_type']}")
        print(f"     Gap: {result['gap']:.4f} (distributions overlap)")
        if result['overlap_start'] is not None:
            print(f"     Overlap Region: [{result['overlap_start']:.4f} → {result['overlap_end']:.4f}]")
        
        # Suggest conservative thresholds based on percentiles
        if result['is_distance']:
            # For distance: use P95 of matching (captures 95% of matches)
            conservative_threshold = result['same_p95']
            aggressive_threshold = result['same_p99']
            print(f"     💡 Conservative Threshold (95% Recall): {score} <= {conservative_threshold:.4f}")
            print(f"     💡 Aggressive Threshold (99% Recall): {score} <= {aggressive_threshold:.4f}")
        else:
            # For similarity: use MIN for 100% recall or P50 for balanced approach
            min_threshold = result['same_min']
            balanced_threshold = result['same_p50']
            print(f"     💡 Maximum Recall (100%): {score} >= {min_threshold:.4f}")
            print(f"     💡 Balanced Threshold (P50): {score} >= {balanced_threshold:.4f}")
        
        overlapping_scores.append(score)

# Summary
print("\n" + "="*100)
print("SUMMARY:")
print("="*100)

if len(mutually_exclusive_scores) > 0:
    print(f"\n✅ MUTUALLY EXCLUSIVE SCORES ({len(mutually_exclusive_scores)}):")
    print(f"   These scores have NO overlap - can achieve 100% P+R individually!")
    for score, op, threshold in mutually_exclusive_scores:
        print(f"   • {score} {op} {threshold}")
    
    print(f"\n🎯 RECOMMENDED WHERE CLAUSE (using any mutually exclusive score):")
    # Use the one with largest gap
    best = builtins.max(comparison_results, key=lambda x: abs(x['gap']) if x['overlap_type'] == "MUTUALLY EXCLUSIVE" else -999)
    print(f"   WHERE {best['score']} {best['threshold_operator']} {best['optimal_threshold']}")
    print(f"   This guarantees 100% Precision + 100% Recall!")

if len(overlapping_scores) > 0:
    print(f"\n⚠️  OVERLAPPING SCORES ({len(overlapping_scores)}):")
    print(f"   These scores have overlap - cannot achieve 100% P+R individually.")
    print(f"   Options:")
    print(f"   1. Use composite weighted score: combine multiple overlapping scores")
    print(f"   2. Use multi-dimensional thresholds: (score1 >= t1 AND score2 >= t2 AND ...)")
    print(f"   3. Accept trade-off: High recall (99%) with some false positives")
    print(f"   4. Train ML classifier: learn complex decision boundaries")

# Create enhanced WHERE clause recommendations
print(f"\n" + "="*100)
print("THRESHOLD REFINEMENT RECOMMENDATIONS:")
print("="*100)

if len(mutually_exclusive_scores) > 0:
    print("\n✅ OPTION 1: Single Mutually Exclusive Score (BEST)")
    best = builtins.max(comparison_results, key=lambda x: abs(x['gap']) if x['overlap_type'] == "MUTUALLY EXCLUSIVE" else -999)
    print(f"   WHERE {best['score']} {best['threshold_operator']} {best['optimal_threshold']}")
    print(f"   Expected: 100% Precision + 100% Recall")

print("\n🔄 OPTION 2: Multi-Dimensional Thresholds (High Recall)")
print("   Combine multiple scores with conservative thresholds (P95 for matching):")
conservative_conditions = []
for result in comparison_results:
    if result['is_distance']:
        threshold = round(result['same_p95'], 4)
        conservative_conditions.append(f"{result['score']} <= {threshold}")
    else:
        # Use P50 (median) for balanced approach
        threshold = round(result['same_p50'], 4)
        if threshold > 0:
            conservative_conditions.append(f"{result['score']} >= {threshold}")

# Use top discriminative scores
name_conds = [c for c in conservative_conditions if 'name' in c.lower()]
geo_conds = [c for c in conservative_conditions if 'geo' in c.lower()]
addr_conds = [c for c in conservative_conditions if 'address' in c.lower()]

multi_dim_where = " AND ".join(
    (name_conds[:2] if name_conds else []) + 
    (geo_conds[:1] if geo_conds else []) +
    (addr_conds[:1] if addr_conds else [])
)
if multi_dim_where:
    print(f"   WHERE {multi_dim_where}")
    print(f"   Expected: ~95-99% Recall with moderate precision")

print("\n🎯 OPTION 3: MIN-Based Strategy (100% Recall Target)")
print("   Use MIN of matching entities to capture ALL same-entity pairs:")
min_based_conditions = []
for result in comparison_results:
    if result['is_distance']:
        # For distance: use MAX of matching (captures all matches)
        threshold = round(result['same_max'] * 1.01, 4)  # Slight buffer
        min_based_conditions.append(f"{result['score']} <= {threshold}")
    else:
        # For similarity: use MIN of matching (captures all matches)
        threshold = round(result['same_min'] * 0.99, 4) if result['same_min'] > 0 else 0.0
        if threshold > 0:
            min_based_conditions.append(f"{result['score']} >= {threshold}")

# Strategy 3a: Use all MIN thresholds with AND (most restrictive)
name_min = [c for c in min_based_conditions if 'name' in c.lower()]
geo_min = [c for c in min_based_conditions if 'geo' in c.lower()]
addr_min = [c for c in min_based_conditions if 'address' in c.lower()]

min_where_and = " AND ".join((name_min[:2] if name_min else []) + (geo_min[:1] if geo_min else []))
if min_where_and:
    print(f"   3a) AND combination (name + geo):")
    print(f"       WHERE {min_where_and}")
    print(f"       Expected: 100% Recall guaranteed, precision depends on overlap")

# Strategy 3b: Use MIN thresholds with OR (most permissive)
min_where_or = " OR ".join((name_min[:1] if name_min else []) + (geo_min[:1] if geo_min else []))
if min_where_or:
    print(f"   3b) OR combination (name OR geo):")
    print(f"       WHERE {min_where_or}")
    print(f"       Expected: 100% Recall guaranteed, may have many false positives")

print("\n💡 OPTION 4: MAX-Based Strategy (High Precision Target)")
print("   Use MAX of non-matching entities to minimize false positives:")
max_based_conditions = []
for result in comparison_results:
    if result['is_distance']:
        # For distance: use MIN of non-matching (rejects most non-matches)
        threshold = round(result['diff_min'] * 0.99, 4) if result['diff_min'] > 0 else 0.0
        if threshold > 0:
            max_based_conditions.append(f"{result['score']} <= {threshold}")
    else:
        # For similarity: use MAX of non-matching (rejects most non-matches)
        threshold = round(result['diff_max'] * 1.01, 4)
        max_based_conditions.append(f"{result['score']} >= {threshold}")

name_max = [c for c in max_based_conditions if 'name' in c.lower()]
geo_max = [c for c in max_based_conditions if 'geo' in c.lower()]
addr_max = [c for c in max_based_conditions if 'address' in c.lower()]

max_where = " OR ".join((name_max[:2] if name_max else []) + (geo_max[:1] if geo_max else []))
if max_where:
    print(f"   WHERE {max_where}")
    print(f"   Expected: High precision, but may miss some same-entity pairs (lower recall)")

print("\n🔬 OPTION 5: Hybrid MIN/MAX Strategy")
print("   Combine MIN (for recall) and MAX (for precision) thresholds:")

# Find scores with smallest gaps (most overlap)
sorted_by_gap = sorted(comparison_results, key=lambda x: abs(x['gap']))
most_overlapping = sorted_by_gap[:3]  # Top 3 most overlapping scores

hybrid_conditions = []
for result in most_overlapping:
    if result['is_distance']:
        # Use average of same_max and diff_min
        threshold = round((result['same_max'] + result['diff_min']) / 2, 4)
        hybrid_conditions.append(f"{result['score']} <= {threshold}")
    else:
        # Use average of same_min and diff_max
        threshold = round((result['same_min'] + result['diff_max']) / 2, 4)
        if threshold > 0:
            hybrid_conditions.append(f"{result['score']} >= {threshold}")

if hybrid_conditions:
    print(f"   Using top 3 most discriminative (overlapping) scores:")
    for i, result in enumerate(most_overlapping):
        print(f"   {i+1}. {result['score']} (gap: {result['gap']:.4f})")
    
    hybrid_where = " AND ".join(hybrid_conditions)
    print(f"   WHERE {hybrid_where}")
    print(f"   Expected: Balance between precision and recall")

print("\n🎯 OPTION 6: Aggressive MIN Strategy (Maximum Recall)")
print("   Use lowest thresholds (MIN or P99) to capture maximum matches:")
aggressive_conditions = []
for result in comparison_results:
    if result['is_distance']:
        threshold = round(result['same_p99'], 4)
        aggressive_conditions.append(f"{result['score']} <= {threshold}")
    else:
        # Use MIN for absolute maximum recall
        threshold = round(result['same_min'] * 0.98, 4) if result['same_min'] > 0 else 0.0
        if threshold > 0:
            aggressive_conditions.append(f"{result['score']} >= {threshold}")

# Use top discriminative scores
name_conds_agg = [c for c in aggressive_conditions if 'average_name' in c.lower()][:1]
geo_conds_agg = [c for c in aggressive_conditions if 'geo' in c.lower()][:1]

aggressive_where = " OR ".join(name_conds_agg + geo_conds_agg)
if aggressive_where:
    print(f"   WHERE {aggressive_where}")
    print(f"   Expected: ~100% Recall, but likely many false positives")

print("\n✓ Analysis complete! Use these insights to refine your WHERE clause.")


## 5. Grid Search Optimizer for Complex Rule Discovery

To find the optimal combination of score thresholds, we'll perform a grid search. This involves programmatically generating and testing thousands of complex `WHERE` clauses to find the one that maximizes precision and recall.

**Methodology:**
1.  **Generate Threshold Candidates**: For each score, we'll use its statistical distribution (P50, P90, P95, P99 of matching entities) to create a set of potential threshold values.
2.  **Create Rule Combinations**: We'll combine 2 to 4 different scores together using `AND` logic.
3.  **Iterate and Evaluate**: The grid search will iterate through every possible combination of scores and their threshold candidates, evaluating the precision and recall for each generated `WHERE` clause.
4.  **Identify Best Rule**: The combination that achieves the highest score (prioritizing 100% recall, then highest precision) will be identified as the optimal rule.

In [ ]:
import itertools
import time

print("="*100)
print("DOMAIN-BASED GRID SEARCH OPTIMIZER")
print("="*100)

# 1. Define Score Domains
score_domains = {
    'name_scores': [
        'name_score_jaccard', 'name_score_lcs', 'name_score_levenshtein', 
        'name_score_sbert', 'name_score_containment', 'average_name_score'
    ],
    'normalized_name_scores': [
        'normalized_name_score_jaccard', 'normalized_name_score_lcs', 'normalized_name_score_levenshtein',
        'normalized_name_score_sbert', 'normalized_name_score_containment', 'average_normalized_name_score'
    ],
    'individual_scores': [
        'star_ratings_score', 'property_type_score', 'name_unit_score', 
        'address_unit_score', 'supplier_score'
    ]
}

# Flatten list of all scores to be used
all_candidate_scores = [score for domain in score_domains.values() for score in domain]
print(f"✓ Defined {len(score_domains)} score domains with a total of {len(all_candidate_scores)} candidate scores.")

# 2. Generate threshold candidates for each score
# We use the statistics from the 'comparison_results' calculated in the previous cell
threshold_candidates = {}
for score in all_candidate_scores:
    stats = next((s for s in comparison_results if s['score'] == score), None)
    if stats:
        # For all these scores, higher is better (similarity)
        candidates = [stats['same_min'], stats['same_p50'], stats['same_p90'], stats['same_p95']]
        
        # Add a mid-point threshold for balance
        mid_point = (stats['same_min'] + stats['same_max']) / 2
        candidates.append(mid_point)
        
        threshold_candidates[score] = sorted(list(set(round(c, 4) for c in candidates if c > 0))) # Only use thresholds > 0

print("✓ Generated threshold candidates for each score.")

# 3. Initialize Grid Search
best_rule = None
best_metrics = {'recall': 0, 'precision': -1}
rules_tested = 0
start_time = time.time()

ground_truth_same_count = same_entity_df.count()
ground_truth_diff_count = diff_entity_df.count()

# --- Helper function to generate rule permutations for a domain ---
def get_domain_rules(domain_name, scores, num_ors=2):
    domain_rules = ["1=1"] # Start with a pass-through rule
    
    # Generate combinations of scores within the domain
    for r in range(1, num_ors + 1):
        score_combinations = itertools.combinations(scores, r)
        for score_combo in score_combinations:
            
            # Get all threshold combinations for the selected scores
            threshold_combos = [threshold_candidates.get(s, []) for s in score_combo]
            threshold_permutations = itertools.product(*threshold_combos)

            for thresholds in threshold_permutations:
                conditions = [f"{score_name} >= {thresholds[i]}" for i, score_name in enumerate(score_combo)]
                domain_rules.append(f"({ ' OR '.join(conditions) })")
    return domain_rules

# --- Generate all possible sub-rules for each domain ---
print("\nGenerating sub-rules for each domain...")
name_domain_rules = get_domain_rules('name_scores', score_domains['name_scores'], num_ors=2)
norm_name_domain_rules = get_domain_rules('normalized_name_scores', score_domains['normalized_name_scores'], num_ors=2)
individual_domain_rules = get_domain_rules('individual_scores', score_domains['individual_scores'], num_ors=1) # Only single conditions for these
print(f"  - Name Domain: {len(name_domain_rules)} rules")
print(f"  - Normalized Name Domain: {len(norm_name_domain_rules)} rules")
print(f"  - Individual Score Domain: {len(individual_domain_rules)} rules")


# --- Combine domain rules and evaluate ---
print("\nStarting grid search by combining domain rules...")
all_rule_permutations = itertools.product(name_domain_rules, norm_name_domain_rules, individual_domain_rules)

for rule_tuple in all_rule_permutations:
    rules_tested += 1
    
    # Combine domain rules with AND
    final_conditions = [rule for rule in rule_tuple if rule != "1=1"]
    if not final_conditions:
        continue
        
    where_clause = " AND ".join(final_conditions)

    # Evaluate the rule
    try:
        filtered_df = df.filter(where_clause)
        total_captured = filtered_df.count()

        if total_captured == 0:
            continue

        same_captured = filtered_df.filter(col("id_i") == col("id_j")).count()
        
        recall = 100 * same_captured / ground_truth_same_count if ground_truth_same_count > 0 else 0
        
        # Optimization Goal: Must have 100% recall.
        if recall == 100:
            diff_captured = total_captured - same_captured
            precision = 100 * same_captured / total_captured if total_captured > 0 else 0
            
            # Among those with 100% recall, find the one with the highest precision.
            if precision > best_metrics['precision']:
                best_metrics = {'recall': recall, 'precision': precision, 'total_captured': total_captured, 'same_captured': same_captured, 'diff_captured': diff_captured}
                best_rule = where_clause
                print(f"Found new best rule! P: {precision:.2f}%, R: {recall:.2f}% -> {where_clause}")

    except Exception as e:
        continue # Ignore rules that cause Spark errors

end_time = time.time()
print(f"\n\n{'='*100}")
print("GRID SEARCH COMPLETE")
print("="*100)
print(f"Total rules tested: {rules_tested:,}")
print(f"Total time: {end_time - start_time:.2f} seconds")

if best_rule:
    print("\n🏆 BEST RULE FOUND (100% RECALL):")
    print(f"   WHERE {best_rule}")
    print("\n📊 METRICS:")
    print(f"   Recall: {best_metrics['recall']:.2f}%")
    print(f"   Precision: {best_metrics['precision']:.2f}%")
    print("\n📈 CAPTURED:")
    print(f"   Total: {best_metrics['total_captured']:,}")
    print(f"   ├─ True Positives (same-entity): {best_metrics['same_captured']:,} / {ground_truth_same_count:,}")
    print(f"   └─ False Positives (diff-entity): {best_metrics['diff_captured']:,}")
else:
    print("\n⚠️ No rule found that achieves 100% recall with the given score domains and thresholds.")
    print("Consider adjusting the threshold candidates or allowing more OR conditions.")
